In [ ]:
from google.colab import drive
drive.mount('/content/drive')

folder_path = "/content/drive/MyDrive/WTIND"

Mounted at /content/drive


In [ ]:
!pip install easyocr sentence-transformers faiss-cpu pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 27.2 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import easyocr
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer

In [ ]:
image_paths = []
for file in os.listdir(folder_path):
    if file.endswith((".png", ".jpg", ".jpeg", ".tif")):
        image_paths.append(os.path.join(folder_path, file))

print("Total images found:", len(image_paths))


Total images found: 932


In [ ]:
def convert_tif_to_png(path):
    img = cv2.imread(path)
    new_path = path.replace(".tif", ".png")
    cv2.imwrite(new_path, img)
    return new_path


In [ ]:

# =========================
# 5. OCR (TEXT EXTRACTION)
# =========================
reader = easyocr.Reader(['en'], gpu=True)


image_texts = []

for path in image_paths:
    try:

        if path.endswith(".tif"):
            path = convert_tif_to_png(path)

        result = reader.readtext(path, detail=0)
        text = " ".join(result)

        image_texts.append({
            "image": path,
            "text": text
        })

    except Exception as e:
        print("Error processing:", path)
        print(e)

print("OCR completed:", len(image_texts))

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteOCR completed: 932


In [ ]:
# =========================
# 6. CLEAN TEXT
# =========================
def clean_text(text):
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    return text

for item in image_texts:
    item["text"] = clean_text(item["text"])

In [ ]:
# =========================
# 7. CHUNK TEXT
# =========================
def chunk_text(text, chunk_size=150):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        if chunk.strip() != "":
            chunks.append(chunk)

    return chunks


documents = []

for item in image_texts:
    text = item["text"]

    if text.strip() == "":
        continue

    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):
        documents.append({
            "text": chunk,
            "image": item["image"],
            "chunk_id": i
        })

print("Total chunks:", len(documents))

Total chunks: 2342


In [ ]:
print(documents[0])

{'text': 'MAKiNG THE BEST USE OF THIS BOOK T Anyone who knows how to read can use this book in his Own home. Even those who do not read can Iearn 93 from the pictures. But to make the fullest ad best use of the book, people often need some instruction; This 8 can be done in several ways; health worker or ayone who gives out the book should make sure that people understand how to use the list of Contents, the Index, the Green Pages ad the Vocabulary. Take special care to give examples of how to look things up. Urge each person to carefully read the sections of the book that will help him understand what may be helpful to what could be harmful or dangerous, and when it is important to gt help (see especially Chapters 1, 2, 6, ad 9, and also the SIGNS OF DANGEROUS ILLNESS,', 'image': '/content/drive/MyDrive/WTIND/00000045.png', 'chunk_id': 0}


In [ ]:


# =========================
# 8. EMBEDDINGS
# =========================
model = SentenceTransformer('all-MiniLM-L6-v2',device='cuda')

texts = [doc["text"] for doc in documents]

print("Texts for embedding:", len(texts))

embeddings = model.encode(texts)

embeddings = np.array(embeddings)


if len(embeddings.shape) == 1:
    embeddings = embeddings.reshape(1, -1)

print("Embeddings shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Texts for embedding: 2342
Embeddings shape: (2342, 384)


In [ ]:
# =========================
# 9. FAISS INDEX
# =========================
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index created!")

FAISS index created!


In [ ]:
# =========================
# 10. SAVE FILES
# =========================
faiss.write_index(index, "/content/index.faiss")

with open("/content/metadata.pkl", "wb") as f:
    pickle.dump(documents, f)

print("Files saved successfully!")

Files saved successfully!


In [ ]:
print("Images:", len(image_paths))
print("OCR texts:", len(image_texts))
print("Chunks:", len(documents))
print("Embeddings:", embeddings.shape)

Images: 932
OCR texts: 932
Chunks: 2342
Embeddings: (2342, 384)


In [ ]:
import easyocr

reader = easyocr.Reader(['en'], gpu=False)
image_texts = []

for path in image_paths:
    result = reader.readtext(path, detail=0)
    text = " ".join(result)
    image_texts.append({
        "image": path,
        "text": text
    })

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

ValueError: <COMPRESSION.CCITTFAX4: 4> requires the 'imagecodecs' package

In [ ]:
print("Images:", len(image_paths))
print("Texts extracted:", len(image_texts))
print("Documents:", len(documents))
print("Embeddings shape:", embeddings.shape)

Images: 0
Texts extracted: 0
Documents: 0
Embeddings shape: (0,)


In [ ]:
def clean_text(text):
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    return text

for item in image_texts:
    item["text"] = clean_text(item["text"])

In [ ]:
def chunk_text(text, chunk_size=300):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks


documents = []

for item in image_texts:
    chunks = chunk_text(item["text"])

    for chunk in chunks:
        documents.append({
            "text": chunk,
            "image": item["image"],
            "chunk_id": i
        })

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [doc["text"] for doc in documents]

embeddings = model.encode(texts)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
print("Embeddings shape:", embeddings.shape)

Embeddings shape: (0,)


In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

IndexError: tuple index out of range

In [ ]:
import pickle

# save FAISS index
faiss.write_index(index, "index.faiss")

# save metadata (text + image path)
with open("metadata.pkl", "wb") as f:
    pickle.dump(documents, f)